# Reproduce: `desc_seed`

**Axis:** description-seeded reference at k=0 (Chapter C)

Recomputes all scoring from **frozen** llama3.2 predictions and the observed 53×22 matrix.
Predictions are **not** regenerated (Ollama output is not guaranteed reproducible across versions even at temperature 0).

```bash
cd REPO_ROOT
.venv/bin/python -m abrg.validate_reproduce --run-dir abrg/output/desc_seed
```

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

CWD = Path.cwd().resolve()
REPO_ROOT = CWD
for _ in range(8):
    if (REPO_ROOT / "abrg" / "__init__.py").is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise RuntimeError("could not locate repo root (abrg/__init__.py)")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from abrg.desc_seed.metrics import recompute_all_metrics
from abrg.desc_seed.validate import compare_metrics

RUN_DIR = CWD if (CWD / "reproduce_config.json").is_file() else (REPO_ROOT / "abrg/output/desc_seed").resolve()
CFG = json.loads((RUN_DIR / "reproduce_config.json").read_text(encoding="utf-8"))
print("RUN_DIR:", RUN_DIR)
print("axis:", CFG.get("axis"))
print("frozen_inputs_policy:", CFG.get("frozen_inputs_policy"))

In [ ]:
actual = recompute_all_metrics(RUN_DIR)
REPRO_DIR = RUN_DIR / "nb_repro"
REPRO_DIR.mkdir(parents=True, exist_ok=True)
(REPRO_DIR / "scoring_results.json").write_text(
    json.dumps(actual, indent=2) + "\n", encoding="utf-8"
)
print("n_apps:", actual["n_apps"])
print("self_cross cosine temp0:", actual["self_cross"]["cosine_temp00_auc"])
print("within_app prior median:", actual["within_app"]["prior_median"])

In [ ]:
report = compare_metrics(CFG["expected"], actual, atol=float(CFG.get("atol", 0.001)))
report["mode"] = "notebook"
(RUN_DIR / "nb_validate_report.json").write_text(
    json.dumps(report, indent=2) + "\n", encoding="utf-8"
)

mismatch_rows = [r for r in report["rows"] if not r["match"]]
summary = {
    "ok": report["ok"],
    "n_metrics": report["n_metrics"],
    "n_mismatch": report["n_mismatch"],
    "mismatches": report["mismatches"],
}
print(json.dumps(summary, indent=2))
if mismatch_rows:
    print("\nFirst mismatches:")
    for row in mismatch_rows[:10]:
        print(row)
print("\nVALIDATE:", "OK" if report["ok"] else "MISMATCH (see nb_validate_report.json)")